# MediXplain - EDA and Model Comparison

Author: **Mithin Sagar S**  
GitHub: [github.com/mithinsagar](https://github.com/mithinsagar)  
Project: MediXplain - Explainable Disease Prediction System

This notebook explores the base Kaggle dataset, compares the scikit-learn model zoo, and previews the FAISS RAG retrieval output. It is intended as a demonstration companion to the pipeline scripts under `ml/` and `rag/`.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.figsize'] = (9, 5)

## 2. Load the Cleaned Dataset

In [ ]:
from config import CONFIG

df = pd.read_csv(CONFIG.paths.processed / 'disease_dataset_clean.csv')
target_col = df.columns[0]
print('Shape:', df.shape)
print('Target column:', target_col)
df.head()

## 3. Class Distribution

In [ ]:
counts = df[target_col].value_counts().head(20)
counts.plot(kind='barh', color='#3A8EFD')
plt.gca().invert_yaxis()
plt.title('Top-20 diseases by row count')
plt.xlabel('rows')
plt.show()

## 4. Symptom Density per Row

In [ ]:
feature_cols = [c for c in df.columns if c not in (target_col, 'symptom_text')]
density = df[feature_cols].sum(axis=1)
density.hist(bins=40, color='#2FD6A4')
plt.title('Number of active symptoms per row')
plt.xlabel('active symptoms')
plt.ylabel('rows')
plt.show()

## 5. Model Benchmark Results

In [ ]:
results_path = CONFIG.paths.models / 'model_results.csv'
if results_path.exists():
    results = pd.read_csv(results_path)
    display(results.sort_values('accuracy', ascending=False))
    ax = results.plot(kind='bar', x='model', y='accuracy', legend=False, color='#8B7FFF')
    ax.set_title('Model accuracy on held-out test set')
    ax.set_ylabel('accuracy')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print('Run `python -m ml.train_models` first to populate models/model_results.csv')

## 6. RAG Retrieval Preview

In [ ]:
try:
    from rag.retriever import load_retriever
    retriever = load_retriever()
    query_disease = df[target_col].iloc[0]
    query_symptoms = [c.replace('_', ' ') for c in feature_cols if df.iloc[0][c] == 1][:5]
    for hit in retriever.retrieve(query_disease, query_symptoms, top_k=3):
        print(f"[{hit['score']:.3f}] {hit['disease']}\n{hit['paragraph']}\n")
except Exception as exc:
    print('RAG index not built yet:', exc)